# Context-Deference — Phase 2: causal steering & subspace removal

Turns the correlational cosine result into a **causal** one. Three checks:

1. **Necessity** — ablating a behavior's suppression direction should *reduce* its override rate.
2. **Random-direction baseline** — a random matched-norm direction should *not* reproduce that
   effect (else the direction isn't special).
3. **Subspace-removal consistency** (arXiv:2509.21305) — remove behavior *B*'s suppression subspace
   and measure every behavior. Diagonal collapses + off-diagonal stable ⇒ **distinct/separable**
   suppressors; everything collapsing together ⇒ a **shared** suppressor.

Needs a GPU. Reuses `src/pipeline.py` so extraction matches the MVP driver exactly.

In [ ]:
# === Colab / remote-GPU bootstrap — no-op when run locally ===
# On Colab: uploads context-deference.zip (code + data), installs deps, logs into HF (gated models).
import sys, os
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/suppression"):
        from google.colab import files
        print("Select context-deference.zip to upload ...")
        files.upload()
        os.system("unzip -oq context-deference.zip -d /content")
    os.chdir("/content/suppression/notebooks")            # notebooks read ../configs, ../data, ../src
    os.system("pip -q install -r ../requirements.txt")
    from huggingface_hub import login                      # gated Llama-3.1-8B needs a token:
    login()                                                #   accept the license on HF, paste token here
    print("bootstrap complete | cwd:", os.getcwd())
else:
    print("local run — Colab bootstrap skipped")

In [ ]:
# On Colab: !pip install -q -r ../requirements.txt
import sys, os
sys.path.append(os.path.abspath(".."))
import numpy as np, torch
from src import model as M, data as D, pipeline as PL, steering as St, eval as Ev

torch.set_grad_enabled(False)
# Env overrides (same as the driver): defaults = science run; for a local check e.g.
#   CD_MODEL=qwen2.5-1.5b-instruct CD_MAX_PAIRS=16 CD_MAX_NEW_TOKENS=24 CD_N_RANDOM=4 CD_CONTRAST=manip_vs_clean
cfg        = D.load_behaviors_config("../configs/behaviors.yaml")
models_cfg = D.load_yaml("../configs/models.yaml")
# CD_MODEL may be a models.yaml key OR any transformer_lens name (auto-resolved -> mid-stack layer).
mc  = M.resolve_model_cfg(models_cfg, os.environ.get("CD_MODEL", models_cfg["default_model"]))
POS = mc.get("default_position", -1)
LAYER = mc.get("default_layer")            # None if auto-resolved -> filled after the model loads
CONTRAST     = os.environ.get("CD_CONTRAST", cfg["contrast"]["mode"])   # CD_CONTRAST to A/B a contrast
SUBSET       = os.environ["CD_SUBSET"].split(",") if os.environ.get("CD_SUBSET") else cfg["mvp_subset"]
N_EVAL       = int(os.environ.get("CD_MAX_PAIRS", "48"))     # prompts/behavior under intervention
MAX_NEW_TOKENS = int(os.environ.get("CD_MAX_NEW_TOKENS", "64"))
N_RANDOM     = int(os.environ.get("CD_N_RANDOM", "8"))
print("model:", mc["tl_name"], "| contrast:", CONTRAST, "| behaviors:", SUBSET, "| N_EVAL:", N_EVAL, "| N_RANDOM:", N_RANDOM)

In [ ]:
bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype", "bfloat16"))
LAYER = M.default_layer_for(mc, bundle)    # fill mid-stack (n_layers//2) if the model was auto-resolved
print("loaded", bundle.name, "| layers:", bundle.n_layers, "| d_model:", bundle.d_model, "| LAYER:", LAYER)

In [ ]:
# --- extract suppression directions + manip prompts (one GPU pass/behavior, shared pipeline) ---
sup_dirs, prompts_by, pairs_by = {}, {}, {}
for b in SUBSET:
    print("extracting:", b)
    ex = PL.extract_behavior(bundle, b, cfg, layer=LAYER, position=POS, contrast_mode=CONTRAST,
                             max_items=N_EVAL, max_new_tokens=MAX_NEW_TOKENS)
    sup_dirs[b]   = ex.suppression_dir
    prompts_by[b] = ex.prompts["manip"]
    pairs_by[b]   = ex.pairs
    M.free()
# 1-D subspace to remove per behavior = its suppression direction (widen with residual/PCs if desired)
subspace_by = {b: sup_dirs[b].vec[None, :] for b in SUBSET}

In [ ]:
# --- (1) necessity + (2) random-direction baseline ---
# ablate the suppression direction across all layers; the override rate should fall below baseline
# AND below what random matched-norm directions produce.
for b in SUBSET:
    res = St.random_direction_baseline(bundle, b, sup_dirs[b], prompts_by[b], pairs_by[b],
                                       n_random=N_RANDOM, seed=0, max_new_tokens=MAX_NEW_TOKENS)
    p = Ev.random_baseline_pvalue(res["real_ablated"], res["random_ablated"], "decrease")
    print(f"{b:<20} baseline={res['baseline']:.2f}  real_ablated={res['real_ablated']:.2f}  "
          f"random_mean={np.mean(res['random_ablated']):.2f}  p={p:.3f}")
    M.free()

In [ ]:
# --- (3) subspace-removal consistency matrix ---
cc = St.subspace_removal_consistency_check(bundle, SUBSET, subspace_by, prompts_by, pairs_by,
                                           max_new_tokens=MAX_NEW_TOKENS)
print("baseline override rates:", {k: round(v, 2) for k, v in cc["baseline"].items()})
print("\nrows = removed subspace, cols = measured behavior's override rate")
print(" " * 20 + "".join(f"{b[:12]:>13}" for b in SUBSET))
for rem in SUBSET:
    print(f"{rem:<20}" + "".join(f"{cc['matrix'][rem][meas]:>13.2f}" for meas in SUBSET))
print("\nsummary (own_drop >> other_drop => selective/distinct):")
for b, s in cc["summary"].items():
    print(f"  {b:<20} own_drop={s['own_drop']:+.2f}  other_drop={s['other_drop']:+.2f}  selective={s['selective']}")

## Reading the causal result

- **Necessity + baseline:** `real_ablated` well below `baseline` and below `random_mean` (small `p`)
  ⇒ that suppression direction is *causally* responsible for the override — not any direction will do.
- **Consistency matrix:** the diagonal is the override rate of a behavior after removing *its own*
  suppression subspace; off-diagonal is a behavior's rate after removing *another's*.
  - Diagonal collapses, off-diagonal ≈ baseline ⇒ **distinct/separable** suppressors (`selective=True`).
  - Removing any one subspace collapses all ⇒ a **shared** suppressor.
  - Partial cross-effects ⇒ **distinct-but-shared-knob** — corroborate with the residual-cosine PCA
    (`subspace.shared_knob_test`) and principal angles from the MVP driver.

Together with the correlational matrices this is the full pre-registered read: geometry (cosines/PCA)
+ causality (ablation necessity, random baseline, subspace-removal). A null (distinct) remains a result.